# LeetCode 3602 — Hexadecimal and Hexatrigesimal Conversion bằng Python

Link đề: https://leetcode.com/problems/hexadecimal-and-hexatrigesimal-conversion/ · Độ khó: **Easy** · Chủ đề: Math, String

Skeleton LeetCode:

```python
class Solution:
    def concatHex36(self, n: int) -> str:
        ...
```

> Ghi chú: tên bài, đề, ví dụ và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `n`, tính `n²` rồi viết ở **hệ 16** (chữ số `0-9`, `A-F`), tính `n³` rồi viết ở **hệ 36** (chữ số `0-9`, `A-Z`), sau đó **nối** hai chuỗi (hex trước, base-36 sau).
2. Nắm thuật toán **đổi cơ số bằng phép chia lặp**: lấy `x % base` làm chữ số thấp nhất, rồi `x //= base`, cuối cùng **đảo ngược** các chữ số.
3. Biết dùng bảng chữ số `"0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"` để đổi số `0..35` thành ký tự, và dùng `format(x, "X")` / `int(s, base)` để kiểm tra chéo.
4. Viết 4 cách (dùng `format` cho hex + tự đổi base 36, chia lặp nối chuỗi, chia lặp rồi đảo danh sách, đệ quy) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: viết chữ **thường**, quên **đảo ngược**, dùng cơ số 16 cho cả hai phần, và nhầm `n*2`, `n*3` với `n**2`, `n**3`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`.

1. Tính `n²`, viết ở hệ **thập lục phân** (hexadecimal, cơ số 16) dùng chữ số `0-9` và chữ hoa `A-F`.
2. Tính `n³`, viết ở hệ **tam thập lục phân** (hexatrigesimal, cơ số 36) dùng chữ số `0-9` và chữ hoa `A-Z` (`A = 10`, ..., `Z = 35`).
3. Trả về chuỗi **nối** kết quả (1) rồi (2).

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `13` | `"A91P1"` | `13² = 169 = 10*16 + 9` -> `"A9"`; `13³ = 2197 = 1*36² + 25*36 + 1` -> `"1P1"` |
| 2 | `36` | `"5101000"` | `36² = 1296 = 5*256 + 1*16 + 0` -> `"510"`; `36³ = 46656 = 36³` -> `"1000"` |

Constraints: `1 <= n <= 1000`.

In [1]:
n = 13
sq, cu = n ** 2, n ** 3
print("n^2 =", sq, "| hex  :", format(sq, "X"))
print("n^3 =", cu, "| base36 (kiểm tra ngược):", "1P1", "->", int("1P1", 36))
n = 36
print(n ** 2, format(n ** 2, "X"), n ** 3, int("1000", 36))

n^2 = 169 | hex  : A9
n^3 = 2197 | base36 (kiểm tra ngược): 1P1 -> 2197
1296 510 46656 46656


## 3. Kiến thức nền

### 3.1. Hệ cơ số `b` và "chữ số"

Số `169` ở hệ 10 nghĩa là `1*100 + 6*10 + 9`. Ở hệ cơ số `b`, mỗi vị trí có trọng số `b^k` và chữ số nằm trong `0..b-1`. Hệ 16 cần 16 ký hiệu nên mượn thêm `A..F` cho `10..15`. Hệ 36 cần 36 ký hiệu: `0-9` rồi `A-Z` (`A=10`, `B=11`, ..., `Z=35`).

In [2]:
DIGITS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
print(len(DIGITS), DIGITS[10], DIGITS[15], DIGITS[25], DIGITS[35])
# 169 = 10*16 + 9 -> chữ số 10 là 'A', chữ số 9 là '9'
print(DIGITS[10] + DIGITS[9])

36 A F P Z
A9


### 3.2. `divmod` và đổi cơ số bằng phép chia lặp

`divmod(x, b)` trả cả **thương** và **dư** một lần: `q, r = divmod(x, b)`. Dư `r` chính là **chữ số hàng đơn vị** của `x` ở cơ số `b`; thương `q` là phần còn lại. Lặp lại cho `q` đến khi `x == 0`. Các chữ số lấy ra theo thứ tự **từ thấp lên cao**, nên cuối cùng phải **đảo ngược**.

In [3]:
x, b = 2197, 36
steps = []
while x > 0:
    x, r = divmod(x, b)
    steps.append(r)
print(steps)                  # [1, 25, 1]  (từ thấp lên cao)
print(steps[::-1])            # [1, 25, 1]  (từ cao xuống thấp)
x = 169
out = []
while x:
    x, r = divmod(x, 16)
    out.append(DIGITS[r])
print(out, "->", "".join(reversed(out)))

[1, 25, 1]
[1, 25, 1]
['9', 'A'] -> A9


### 3.3. `format(x, "X")` và `int(s, base)`

- `format(x, "X")` hoặc `f"{x:X}"` cho hex **chữ hoa** (`"x"` thường cho chữ thường).
- `int(s, base)` đọc chuỗi `s` ở cơ số `base` (2..36) về số nguyên, chấp nhận cả chữ hoa lẫn chữ thường. Đây là công cụ **kiểm tra chéo** rất tiện: đổi xong rồi `int(kết_quả, base)` phải bằng số ban đầu.

In [4]:
print(format(169, "X"), format(169, "x"), f"{255:X}")
print(int("A9", 16), int("a9", 16), int("1P1", 36), int("1p1", 36))
print(int("ZZ", 36), 35 * 36 + 35)

A9 a9 FF
169 169 2197 2197
1295 1295


### 3.4. Đảo chuỗi / danh sách và nối chuỗi

`"".join(list)` ghép các ký tự. `reversed(lst)`, `lst[::-1]` hoặc `lst.reverse()` đảo thứ tự. Với chuỗi, `s[::-1]` đảo ngược. Cũng có thể **thêm vào đầu** bằng `ch + s` (mỗi lần tạo chuỗi mới, nhưng chuỗi ở đây chỉ dài vài ký tự).

In [5]:
s = "ABC"
print(s[::-1], "".join(reversed(s)), "X" + s)
lst = [1, 2, 3]
lst.reverse()
print(lst)

CBA CBA XABC
[3, 2, 1]


### 3.5. Vì sao hex "ăn" bit rất đẹp, còn base 36 thì không

Vì `16 = 2^4`, mỗi chữ số hex tương ứng đúng **4 bit**: có thể lấy chữ số bằng `x & 15` và dịch `x >>= 4`. Nhưng `36` không phải lũy thừa của 2 nên **bắt buộc** dùng `%` và `//`. Vì vậy ta viết hàm tổng quát dùng `%`/`//` cho mọi cơ số, và dùng phép dịch bit chỉ như một cách **kiểm tra chéo** cho hex.

Số chữ số của `x` ở cơ số `b` là `floor(log_b(x)) + 1`. Ví dụ `169` ở cơ số 16: `log16(169) ~ 1.85` nên có `2` chữ số (`"A9"`).

In [6]:
import math
def hex_by_shift(x):
    out = []
    while x:
        out.append(DIGITS[x & 15])    # x & 15 == x % 16
        x >>= 4                        # x >> 4 == x // 16
    return "".join(reversed(out))

print(hex_by_shift(169), format(169, "X"), hex_by_shift(1296))
print(math.floor(math.log(169, 16)) + 1, len("A9"))
print(math.floor(math.log(2197, 36)) + 1, len("1P1"))

A9 A9 510
2 2
3 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `n = 13`:

```
n^2 = 169
169 : 16 = 10 dư 9    -> chữ số thấp nhất = 9
10  : 16 = 0  dư 10   -> chữ số tiếp = 10 = 'A'
đọc ngược lại: "A9"

n^3 = 2197
2197 : 36 = 61 dư 1   -> 1
61   : 36 = 1  dư 25  -> 25 = 'P'
1    : 36 = 0  dư 1   -> 1
đọc ngược lại: "1P1"

kết quả = "A9" + "1P1" = "A91P1"
```

Quy tắc: **chia lặp lấy dư, rồi đọc dư từ dưới lên**. Hai phần chỉ khác nhau ở cơ số (`16` và `36`) và ở số được đổi (`n²` và `n³`), nên chỉ cần viết **một hàm đổi cơ số** rồi gọi hai lần.

## 5. Cách 1 — Brute Force / baseline

Dùng công cụ có sẵn càng nhiều càng tốt: hex đã có `format(x, "X")`, riêng base 36 Python **không** có định dạng sẵn nên ta tự đổi bằng vòng lặp chia lặp, cộng ký tự vào **đầu** chuỗi kết quả.

Nút thắt: cách này trộn hai kiểu làm (builtin cho hex, tự viết cho base 36) nên khi đề đổi sang cơ số khác (ví dụ 7 và 20) thì phần builtin không dùng lại được. Nhưng với `n <= 1000` thì rất nhanh (`n³ <= 10^9` chỉ có tối đa 6 chữ số base 36).

In [7]:
DIGITS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

class SolutionBuiltin:
    def concatHex36(self, n):
        hex_part = format(n * n, "X")
        x = n ** 3
        s = ""
        while x > 0:
            x, r = divmod(x, 36)
            s = DIGITS[r] + s          # thêm vào đầu -> không cần đảo ngược
        return hex_part + s

print(SolutionBuiltin().concatHex36(13))
print(SolutionBuiltin().concatHex36(36))

A91P1
5101000


**Độ phức tạp:** số chữ số của `x` ở cơ số `b` là `O(log_b x)`. Ở đây `x <= 10^9` nên tối đa `~30` bước (cơ số 2 là xấu nhất, nhưng ta dùng cơ số 16 và 36). Với `n` đã cho, Time `O(log n)`, Space `O(log n)` cho chuỗi kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách bài thành **hai việc nhỏ giống nhau** — "đổi số nguyên dương `x` sang cơ số `b`" — rồi viết đúng **một** hàm `to_base(x, b)` dùng chung. Hàm này chỉ cần một bảng ký tự `DIGITS` và vòng `while x > 0`.

**Quy tắc "nếu… thì…":**

- Nếu `x > 0` -> chữ số kế tiếp là `DIGITS[x % b]`, rồi `x //= b`.
- Các chữ số sinh ra từ **thấp lên cao**; muốn chuỗi đúng thì **đảo ngược** (hoặc chèn vào đầu).
- Vì `n >= 1` nên `n² >= 1` và `n³ >= 1`, vòng lặp luôn chạy ít nhất một lần, **không** có trường hợp `x = 0` cho ra chuỗi rỗng.

Invariant: tại mọi thời điểm, `x_gốc = x_hiện_tại * b^k + (số tạo bởi k chữ số đã lấy)`.

In [8]:
def to_base(x, b):
    out = []
    while x > 0:
        x, r = divmod(x, b)
        out.append(DIGITS[r])
    return "".join(reversed(out))

def concat_hex36(n):
    return to_base(n * n, 16) + to_base(n ** 3, 36)

print(concat_hex36(13), concat_hex36(36), concat_hex36(1), concat_hex36(1000))

A91P1 5101000 11 F4240GJDGXS


## 7. Các cách tiếp theo

Ba cách khác, cùng ý tưởng chia lặp nhưng khác cách tổ chức.

### Cách 2 — Hàm `to_base` tổng quát, nối chuỗi từng chữ số

Một hàm `convert(x, base)` dùng chung cho cả hai phần. Chữ số mới luôn được **chèn vào đầu** chuỗi nên không cần đảo. Dễ đọc, ngắn.

In [9]:
class SolutionRepeatDiv:
    def convert(self, x, base):
        s = ""
        while x > 0:
            s = DIGITS[x % base] + s
            x //= base
        return s

    def concatHex36(self, n):
        return self.convert(n * n, 16) + self.convert(n * n * n, 36)

print(SolutionRepeatDiv().concatHex36(13), SolutionRepeatDiv().concatHex36(36))

A91P1 5101000


**Độ phức tạp:** Time `O(log n)` (số chữ số), Space `O(log n)`.

### Cách 3 — Thu chữ số vào danh sách rồi đảo

Giống cách 2 nhưng chữ số được `append` vào danh sách (rẻ hơn) rồi `reverse()` một lần ở cuối. Đây là mẫu "đổi cơ số" kinh điển, dùng được cả khi số rất lớn.

In [10]:
class SolutionListReverse:
    def to_base(self, x, base):
        digits = []
        while x:
            digits.append(DIGITS[x % base])
            x //= base
        digits.reverse()
        return "".join(digits)

    def concatHex36(self, n):
        return self.to_base(n ** 2, 16) + self.to_base(n ** 3, 36)

print(SolutionListReverse().concatHex36(13), SolutionListReverse().concatHex36(36))

A91P1 5101000


**Độ phức tạp:** Time `O(log n)`, Space `O(log n)`.

### Cách 4 — Đệ quy

`to_base(x, b) = to_base(x // b, b) + DIGITS[x % b]` với điều kiện dừng `x == 0 -> ""`. Chữ số **cao** được tính trước (ở lời gọi sâu hơn) nên thứ tự đúng ngay, không cần đảo. Độ sâu đệ quy chỉ bằng số chữ số (`<= 8`).

In [11]:
class SolutionRecursive:
    def to_base(self, x, base):
        if x == 0:
            return ""
        return self.to_base(x // base, base) + DIGITS[x % base]

    def concatHex36(self, n):
        return self.to_base(n * n, 16) + self.to_base(n ** 3, 36)

print(SolutionRecursive().concatHex36(13), SolutionRecursive().concatHex36(36))

A91P1 5101000


**Độ phức tạp:** Time `O(log n)`, Space `O(log n)` (ngăn xếp đệ quy + chuỗi).

## 8. TRACE từng bước

Trace `n = 13`: hàm in ra mỗi vòng lặp giá trị `x`, thương, dư và chuỗi đang tích lũy.

| Phần | x | x % base | chữ số | x // base | chuỗi (chèn đầu) |
|---|---|---|---|---|---|
| hex `169` | 169 | 9 | `9` | 10 | `9` |
| hex | 10 | 10 | `A` | 0 | `A9` |
| b36 `2197` | 2197 | 1 | `1` | 61 | `1` |
| b36 | 61 | 25 | `P` | 1 | `P1` |
| b36 | 1 | 1 | `1` | 0 | `1P1` |

In [12]:
def trace_convert(x, base):
    print(f"--- đổi {x} sang cơ số {base} ---")
    s = ""
    step = 0
    while x > 0:
        step += 1
        r = x % base
        q = x // base
        s = DIGITS[r] + s
        print(f"vòng {step}: x={x:>5}  x%{base}={r:>2} -> '{DIGITS[r]}'  x//{base}={q:>3}  chuỗi='{s}'")
        x = q
    return s

a = trace_convert(13 ** 2, 16)
b = trace_convert(13 ** 3, 36)
print("kết quả:", a + b)

--- đổi 169 sang cơ số 16 ---
vòng 1: x=  169  x%16= 9 -> '9'  x//16= 10  chuỗi='9'
vòng 2: x=   10  x%16=10 -> 'A'  x//16=  0  chuỗi='A9'
--- đổi 2197 sang cơ số 36 ---
vòng 1: x= 2197  x%36= 1 -> '1'  x//36= 61  chuỗi='1'
vòng 2: x=   61  x%36=25 -> 'P'  x//36=  1  chuỗi='P1'
vòng 3: x=    1  x%36= 1 -> '1'  x//36=  0  chuỗi='1P1'
kết quả: A91P1


## 9. Code nộp LeetCode

In [13]:
from typing import List, Optional

class Solution:
    def concatHex36(self, n: int) -> str:
        digits = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

        def to_base(x: int, base: int) -> str:
            out = []
            while x > 0:
                out.append(digits[x % base])
                x //= base
            return "".join(reversed(out))

        return to_base(n * n, 16) + to_base(n * n * n, 36)

## 10. Vì sao thuật toán đúng

- **Đổi cơ số đúng:** nếu `x = q*b + r` với `0 <= r < b` thì `r` là chữ số hàng đơn vị duy nhất của biểu diễn cơ số `b` (biểu diễn là duy nhất). Lặp lại cho `q` cho ra các chữ số kế tiếp. Vì `q < x` khi `x > 0`, vòng lặp **luôn dừng**.
- **Thứ tự:** chữ số lấy ra đầu tiên là **thấp nhất** nên danh sách phải đảo.
- **Không chuỗi rỗng:** `n >= 1` kéo theo `n², n³ >= 1`, nên mỗi phần có ít nhất một chữ số.
- **Nối chuỗi:** đề yêu cầu hex trước, base-36 sau; hai chuỗi chỉ ghép, không có ký tự phân cách, và việc tách lại là không cần thiết.
- **Chữ hoa:** bảng `DIGITS` dùng chữ hoa nên khớp yêu cầu `A-F`, `A-Z`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là giá trị đầu vào (`<= 1000`).

| Cách | Time | Space |
|---|---|---|
| Builtin + base 36 tự đổi | `O(log n)` | `O(log n)` |
| Chia lặp nối chuỗi | `O(log n)` | `O(log n)` |
| Danh sách + đảo | `O(log n)` | `O(log n)` |
| Đệ quy | `O(log n)` | `O(log n)` |

Số chữ số hex của `n²` là `~2*log16(n)`, số chữ số base 36 của `n³` là `~3*log36(n)`. Với `n <= 1000`: tối đa `5` chữ số hex (`1000² = 0xF4240`) và `6` chữ số base 36 (`10^9`). Thực tế coi như `O(1)`.

## 12. Bộ test

Cách kiểm tra **độc lập**: tách kết quả theo độ dài của phần hex, rồi đọc ngược bằng `int(..., base)` — phải ra đúng `n²` và `n³`.

In [14]:
impls = [SolutionBuiltin(), SolutionRepeatDiv(), SolutionListReverse(), SolutionRecursive(), Solution()]

def check_roundtrip(res, n):
    k = len(format(n * n, "X"))
    h, t = res[:k], res[k:]
    assert h == h.upper() and t == t.upper(), "phải là chữ hoa"
    assert int(h, 16) == n * n and int(t, 36) == n ** 3

cases = [(13, "A91P1"), (36, "5101000"), (1, "11"), (2, "48"), (10, "64RS")]
for n, exp in cases:
    for impl in impls:
        got = impl.concatHex36(n)
        assert got == exp, (type(impl).__name__, n, got, exp)
print("ví dụ + edge OK")
for n in (1, 1000):
    for impl in impls:
        check_roundtrip(impl.concatHex36(n), n)
print("biên n=1 và n=1000 OK:", Solution().concatHex36(1000))

ví dụ + edge OK
biên n=1 và n=1000 OK: F4240GJDGXS


Kiểm tra `n = 10`: `10² = 100 = 0x64` -> `"64"`; `10³ = 1000 = 27*36 + 28` -> `"RS"` (`R = 27`, `S = 28`). Khớp `"64RS"` ở trên.

In [15]:
# Đối chiếu tất cả các cách + kiểm tra bằng int(..., base) cho MỌI n hợp lệ
for n in range(1, 1001):
    outs = [impl.concatHex36(n) for impl in impls]
    assert len(set(outs)) == 1, (n, outs)
    check_roundtrip(outs[0], n)
print("1..1000: 5 cách giống nhau và round-trip đúng")

1..1000: 5 cách giống nhau và round-trip đúng


In [16]:
import random
random.seed(3602)
for _ in range(300):
    n = random.randint(1, 1000)
    outs = [impl.concatHex36(n) for impl in impls]
    assert len(set(outs)) == 1
    assert outs[0] == format(n * n, "X") + SolutionRepeatDiv().convert(n ** 3, 36)
print("300 test ngẫu nhiên (seed=3602) OK")

300 test ngẫu nhiên (seed=3602) OK


In [17]:
# Đối chiếu độ dài kết quả với công thức số chữ số
for n in range(1, 1001):
    r = Solution().concatHex36(n)
    assert len(r) == len(format(n * n, "X")) + len(SolutionListReverse().to_base(n ** 3, 36))
    assert hex_by_shift(n * n) == format(n * n, "X")
lens = [len(Solution().concatHex36(n)) for n in range(1, 1001)]
print("độ dài kết quả nhỏ nhất/lớn nhất:", min(lens), max(lens))
assert min(lens) == 2 and max(lens) == 11

độ dài kết quả nhỏ nhất/lớn nhất: 2 11


## 13. Các lỗi người mới hay gặp

### Lỗi 1: viết chữ thường

`format(x, "x")` cho chữ **thường**, nhưng đề yêu cầu chữ **hoa**.

In [18]:
def wrong_lower(n):
    return format(n * n, "x") + SolutionRepeatDiv().convert(n ** 3, 36).lower()

print(wrong_lower(13), "!=", Solution().concatHex36(13))
assert wrong_lower(13) != Solution().concatHex36(13)       # a91p1 khác A91P1

a91p1 != A91P1


### Lỗi 2: quên đảo ngược

Chữ số lấy ra theo thứ tự thấp -> cao. Nếu `append` rồi `join` mà **không đảo**, chuỗi bị ngược.

In [19]:
def wrong_no_reverse(n):
    def conv(x, b):
        out = []
        while x:
            out.append(DIGITS[x % b])
            x //= b
        return "".join(out)            # SAI: thiếu reversed
    return conv(n * n, 16) + conv(n ** 3, 36)

print(wrong_no_reverse(13), "!=", Solution().concatHex36(13))
assert wrong_no_reverse(13) != Solution().concatHex36(13)   # 9A1P1 khác A91P1

9A1P1 != A91P1


### Lỗi 3: dùng cơ số 16 cho cả hai phần

Đề đòi `n³` ở **cơ số 36**. Đổi sang 16 cho kết quả khác hẳn.

In [20]:
def wrong_both_hex(n):
    return format(n * n, "X") + format(n ** 3, "X")

print(wrong_both_hex(13), "!=", Solution().concatHex36(13))
assert wrong_both_hex(13) != Solution().concatHex36(13)     # A9895 khác A91P1

A9895 != A91P1


### Lỗi 4: `n * 2`, `n * 3` thay vì `n ** 2`, `n ** 3`

Dấu `*` nhân với 2 chứ không phải **bình phương**. Dễ nhầm khi gõ vội.

In [21]:
def wrong_mul(n):
    return SolutionRepeatDiv().convert(n * 2, 16) + SolutionRepeatDiv().convert(n * 3, 36)

print(wrong_mul(13), "!=", Solution().concatHex36(13))
assert wrong_mul(13) != Solution().concatHex36(13)          # 1A13 khác A91P1

1A13 != A91P1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đổi sang cơ số `b`" -> **chia lặp lấy dư, đảo ngược**. Nhớ bảng `DIGITS`.
- Thấy hex -> thường có sẵn `format(x, "X")` / `hex(x)` (`hex` cho tiền tố `0x`, chữ thường — nhớ cắt và `.upper()`).
- Muốn kiểm tra nhanh: `int(chuỗi, base)` phải trả lại đúng số cũ. Đây là "unit test miễn phí".
- `n ≤ 1000` nên `n³ ≤ 10^9` vừa trong `int` thường; Python không tràn số dù sao.
- Viết **một** hàm `to_base` rồi gọi hai lần, đừng sao chép code hai lần (dễ sai một bản).
- Đừng ghi `x = 0 -> ""` vào kết quả: với bài này không xảy ra, nhưng ở bài khác (`n = 0`) phải trả `"0"`.

## 15. Biến thể của bài

1. Đổi sang cơ số **bất kỳ** `2..36` cho hai lũy thừa tùy ý (`n^a` ở cơ số `b1`, `n^b` ở cơ số `b2`).
2. Cho số âm: thêm dấu `-` rồi đổi trị tuyệt đối (LeetCode 405 *Convert a Number to Hexadecimal* dùng bù hai 32-bit).
3. Đổi **ngược**: cho chuỗi hex/base-36, trả về số nguyên (`int(s, base)` hoặc tự nhân dồn `v = v * base + digit`).
4. Đổi số sang **base -2** (negabinary), LeetCode 1017.
5. Bài "Excel Sheet Column Title" (168) là đổi cơ số 26 **không có chữ số 0**.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Builtin hex + tự base 36 | `format` cho hex, chia lặp cho base 36 | `O(log n)` | `O(log n)` | Làm nhanh khi cơ số là 2/8/16 |
| Chia lặp, chèn đầu | `s = DIGITS[x % b] + s` | `O(log n)` | `O(log n)` | Ngắn, dễ nhớ, chuỗi ngắn |
| Danh sách + đảo | `append` rồi `reverse` | `O(log n)` | `O(log n)` | Mẫu chuẩn, hiệu quả khi chuỗi dài |
| Đệ quy | `f(x//b) + digit` | `O(log n)` | `O(log n)` | Cách nghĩ gọn, dễ chứng minh |

Với `n ≤ 1000` bốn cách đều tức thì; chọn cách bạn **ít sai nhất** (thường là danh sách + đảo hoặc chèn đầu).

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().concatHex36(1)` là gì?

**Bài 2 (tính tay):** `n = 20`: tính `n²` ở hex và `n³` ở base 36 rồi nối.

**Bài 3 (điền TODO):** hoàn thành `to_binary(x)` đổi số nguyên dương sang nhị phân bằng chia lặp.

**Bài 4 (tự viết lại):** viết `from_base(s, base)` đổi chuỗi về số nguyên **không dùng** `int(s, base)`.

**Bài 5 (mini challenge):** viết `to_base_any(x, base)` hỗ trợ cả `x = 0` (trả `"0"`) và `x < 0` (thêm dấu `-`).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().concatHex36(1))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def to_binary(x):
    # TODO: lặp while x > 0, lấy x % 2, x //= 2, nhớ đảo ngược
    return ""

11


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `1² = 1`, `1³ = 1`, nên mỗi phần là `"1"`. Bài 2: `20² = 400`, `20³ = 8000`; chia lặp cho 16 và 36.

**Mức 2 (vừa):** Bài 3: `bits = []`, `bits.append(str(x % 2))`, `x //= 2`, rồi `"".join(reversed(bits))`. Bài 4: `v = 0`, với mỗi ký tự `v = v * base + DIGITS.index(ch)` (nhớ `ch.upper()`). Bài 5: xử lý `x == 0` và dấu trước vòng lặp.

**Mức 3 (gần đáp án):** Bài 2: `400 = 1*256 + 9*16 + 0 = "190"`; dùng chia lặp: `8000 = 6*1296 + 224`, `224 = 6*36 + 8` -> `"668"`. Kết quả `"190668"`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
assert Solution().concatHex36(1) == "11"

# Bài 2: 400 = 0x190 ; 8000 = 6*1296 + 6*36 + 8 -> "668"
assert Solution().concatHex36(20) == "190668"
assert int("190", 16) == 400 and int("668", 36) == 8000

# Bài 3
def to_binary(x):
    bits = []
    while x > 0:
        bits.append(str(x % 2))
        x //= 2
    return "".join(reversed(bits))
assert to_binary(13) == "1101" and to_binary(1) == "1" and to_binary(8) == bin(8)[2:]

# Bài 4
def from_base(s, base):
    v = 0
    for ch in s.upper():
        v = v * base + DIGITS.index(ch)
    return v
assert from_base("1P1", 36) == 2197 and from_base("A9", 16) == 169
assert all(from_base(format(v, "X"), 16) == v for v in range(1, 500))

# Bài 5
def to_base_any(x, base):
    if x == 0:
        return "0"
    sign = "-" if x < 0 else ""
    x = abs(x)
    out = []
    while x:
        out.append(DIGITS[x % base])
        x //= base
    return sign + "".join(reversed(out))
assert to_base_any(0, 16) == "0" and to_base_any(-255, 16) == "-FF" and to_base_any(2197, 36) == "1P1"
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đổi số nguyên dương sang cơ số `b`":**

```python
DIGITS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def to_base(x, b):
    out = []
    while x > 0:
        out.append(DIGITS[x % b])
        x //= b
    return "".join(reversed(out))

return to_base(n * n, 16) + to_base(n ** 3, 36)
```

**Checklist trước khi submit:**

- [ ] Hex dùng chữ **hoa** `A-F`, base 36 dùng `A-Z` (không có chữ thường)?
- [ ] Đã **đảo ngược** các chữ số (hoặc chèn vào đầu)?
- [ ] Phần đầu là `n²` ở cơ số **16**, phần sau là `n³` ở cơ số **36**, nối theo đúng thứ tự?
- [ ] Dùng `**` (lũy thừa), không phải `*` nhân với số mũ?
- [ ] Thử `n = 1` (`"11"`), `n = 13` (`"A91P1"`), `n = 36` (`"5101000"`), `n = 1000`?
- [ ] Chữ ký đúng `concatHex36(self, n: int) -> str`?